In [ ]:

# Fionaa - Financial Loan Application Assessment
# 
# The agent has to research : 

# 1. Examine the contents of the Loan application,  the kind of loan being applied for, and reason or purpose of loan
#    and supporting documentation (bank statements, )
# 3. Bank Policy documents, check applicant meets basic eligibility criteria for loan 
# 4. Lookup Company information in Public Register of Companies (Companies House in UK), l
# 5. Internet searches for online presence of company, website, linked-in. 

# Then evaluate whether

# 1. company is legit, is a going concern, and there are no red flags (bad news stories, or shady directors)
# 2. information is consistant across data sources
# 3. flag any signficant concerns and persist the report to storage

# Additionally we provide ability of the user to dig deeper by:
# 1 Provision of links to source documents, and sub-agents research
# 2 ability to run Q&A on users Annual report (can be 200 pages long!) 


# Design is therefore:

# 1. Planning - what do we need to research - depends on loan application characteristics (not all agents may be required)
# 2. Context management - load one of the loan policy docs required (1 doc per loan type - and they are short so don't really need chuncking and dense vector search)
# 3. Tools - Websearch, LinkedIn search (via MCP server), Companies House Search (via MCP server),
# 4. Specialist SubAgents - Social Media, Companies House,
# 5. Memory - offloading intemediate files to disk and saving applicaation and  final report in Memory Store
# 

In [4]:

%load_ext autoreload
%autoreload 2


In [26]:
import os
import sys
from pathlib import Path
from typing import Literal

import nest_asyncio

nest_asyncio.apply()  

from dotenv import load_dotenv

load_dotenv()


True

In [27]:
from operator import itemgetter

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langgraph.graph import END, START, StateGraph

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Create a workspace directory for our agent
WORKSPACE = Path.cwd().parent / "data" / "workspace"
WORKSPACE.mkdir(exist_ok=True)


In [ ]:
# Add the project root (one level up from notebooks/) to sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from tests.application_forms import steve_application_str

In [28]:
from prompts.agent_prompts import (
    COMPANIES_HOUSE_PROMPT,
    ELIGIBILITY_PROMPT,
    FINANCIAL_ASSESSMENT_PROMPT,
    INTERNET_SEARCH_PROMPT,
    LINKED_IN_PROMPT,
    RESEARCH_PROMPT,
)

In [10]:
from tests.application_forms import steve_application_str

application_str = steve_application_str

# FS  and scrape loan policy docs

In [ ]:
# Bank Policy Docs - 1 time operation -  Save to file-system
# CHANGE TO LANGCHAIN?????

# THIS SHOULD BE AN INIT SCRIPT FOR THE PROPER APP

# converter = DocumentConverter()

# KB_URL = "https://www.fundingoptions.com/knowledge/"
# sources = [
#     "asset-finance", "bridging-loans", "invoice-finance", "merchant-cash-advance", "revolving-credit-facility", "property-development-finance",
#     "business-loans", "working-capital-finance", "short-term-business-loans", "invoice-discounting", "invoice-factoring-vs-invoice-discounting",
#     "loans-for-limited-company", "compare-sole-trader-loans-with-funding-options-by-tide", "finance-leases", "debentures-floating-charges",
#     "trade-finance", "how-to-apply-for-the-growth-guarantee-scheme", "invoice-factoring", "debt-financing", "commercial-property-finance",
#     "find-a-semi-commercial-mortgage-lender", "business-loans-for-women", "unsecured-business-loans", "secured-business-loans", 
#     "business-loans-for-bad-credit", "alternative-finance", "auction-finance", "construction-finance", "equity-finance"
#     ]

# # Scrape the webpages and create the corpus as both text and markdown
# # markdown might provide some extra structure for finding good chunks

# corpus_plain_txt = []

# output_dir = WORKSPACE.parent / "loan_policy_documents"
# corpus_docs = []  # Store the actual DoclingDocument objects
# for source in sources:
#     doc = converter.convert(KB_URL + source).document
#     # Export Markdown format:
#     (output_dir / f"{source}.md").parent.mkdir(parents=True, exist_ok=True)
#     with (output_dir / f"{source}.md").open("w", encoding="utf-8") as fp:
#         fp.write(doc.export_to_markdown())




In [29]:
# file system as a working memory for policy docs- only load documents into context that are relevant to the type of loan
# customer is applying for
# read only because we don't want llm editing or writing over them,

file_system =  FilesystemBackend(
    root_dir=str(WORKSPACE),
    virtual_mode=True
)

# Allowlist of external directories (workspace root + subdirs)
ALLOWED_READS = [
    WORKSPACE,
    WORKSPACE / "loan_policy_documents",
    WORKSPACE / "ocr_output" / "stevejgoodman@gmail.com" # TODO change this
]

def is_allowed(path: Path) -> bool:
    path = path.resolve()
    return any(str(path).startswith(str(root.resolve())) for root in ALLOWED_READS)

@tool
def read_external_file(path: str) -> str:
    """Read a readonly file from an allowed external directory.
    """
    path = path.lstrip("/")
    file_path = Path(WORKSPACE/path)

    if not is_allowed(file_path):
        raise PermissionError(f"Access denied: {file_path}")

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")
    if file_path.is_dir():
        names = sorted(f.name for f in file_path.iterdir() if f.is_file())
        return "Directory; pass a file path. Available files: " + ", ".join(names)

    return file_path.read_text(encoding="utf-8")

In [30]:
# unit test this functionality
eligibility_agent = create_deep_agent(
    model = init_chat_model('claude-sonnet-4-5-20250929', temperature=0),
    tools = [read_external_file],
    backend = file_system,
    system_prompt=ELIGIBILITY_PROMPT
)

In [ ]:
# Test
eligibility_results = eligibility_agent.invoke(
    {
        "messages": [HumanMessage(content=application_str)]
    }
)
print(eligibility_results["messages"][-1].content)


In [22]:
#run a batch of tests
from tests.application_forms import *

In [31]:
# ok but a better extension would be do ragas evals over these
# or use probable fail - run eligibility agent, save agent_output to each app
import tests.application_forms as application_forms

applications = application_forms.get_applications()

for name, app in applications.items():
    print(f"\n===== {name} (expected: {app['expected_outcome']}) =====")
    result = eligibility_agent.invoke({
        "messages": [HumanMessage(content=app["content"])]
    })
    app["agent_output"] = result["messages"][-1].content
    out = app["agent_output"]
    print(out[:500] + "..." if len(out) > 500 else out)


===== steve_application_str (expected: PASS) =====
## Assessment Complete

I have completed a comprehensive eligibility assessment for Steven Goodman's asset finance application. Here's a summary of my findings:

---

### **VERDICT: INELIGIBLE**

The application **does not meet** the eligibility criteria for the requested £10,000 asset finance loan for IT equipment.

---

### **Critical Issues Identified:**

1. **❌ Severely Outdated Bank Statements**
   - Provided statements are from 2015 (approximately 10 years old)
   - Policy requires the mos...

===== synthesia_application_str (expected: FAIL) =====
Perfect! I have completed the comprehensive eligibility assessment. Let me provide you with a summary:

---

## 📋 ELIGIBILITY ASSESSMENT COMPLETE

### **VERDICT: INCONCLUSIVE** ⚠️

---

## Key Findings:

### ✅ **STRENGTHS:**
- **Exceptional cash reserves:** £219.9M at bank
- **Strong net assets:** £197.4M 
- **Established trading history:** 12+ years (since 2012)
- **High revenue:** £5

In [34]:
# with open('../data/eligibility_dataset_assessment.txt', 'wt') as file:
#     file.write(json(applications))


import json
with open('../data/eligibility_dataset_assessment.json', 'wt') as f:
    json.dump(applications, f)

In [ ]:
# check the above details agains the infomration extracted from companies house
financial_assessment_agent = create_deep_agent(
    model = init_chat_model('claude-sonnet-4-5-20250929', temperature=0),
    tools = [read_external_file],
    backend = file_system,
    system_prompt=FINANCIAL_ASSESSMENT_PROMPT
)

In [ ]:
financial_assessment_results = financial_assessment_agent.invoke(
    {
        "messages": [eligibility_results["messages"][-1].content]
    }
)

print(financial_assessment_results["messages"][-1].content)


In [ ]:
# MCP Tools setup

# Linked In Server and Companies House MCP Services
# running these locally for now, and require specific API keys
#  porting them to GCP Cloud Run for Cert Challenge.


import shutil
from pathlib import Path

from langchain_mcp_adapters.client import MultiServerMCPClient

# Run MCP server from project root so it can start correctly (avoids "Connection closed")
project_root = Path.cwd()
try:
    COMPANIES_HOUSE_API_KEY =  os.environ["COMPANIES_HOUSE_API_KEY"]
except KeyError:
    raise ValueError("COMPANIES_HOUSE_API_KEY is not set. Run the cell above and ensure .env in project root contains it.")

# Find npx path - subprocess may not have same PATH as notebook
npx_path = shutil.which("npx") or "/opt/homebrew/bin/npx"

# Ensure PATH includes homebrew bin for subprocess
env = os.environ.copy()
if "/opt/homebrew/bin" not in env.get("PATH", ""):
    env["PATH"] = f"/opt/homebrew/bin:{env.get('PATH', '')}"

ch_client = MultiServerMCPClient(
    {
        "companies-house": {
            "command": npx_path,  # Use full path to npx
            "transport": "stdio",
            "args": ["-y", "companies-house-mcp-server"],
            "env": {
                **env,  # Include PATH in environment
                "COMPANIES_HOUSE_API_KEY": (COMPANIES_HOUSE_API_KEY or "")
            }
        }
    }
)
ch_tools = await ch_client.get_tools()

In [ ]:
# Connect to LinkedIn MCP
# Run the MCP server from its own directory so uv finds the project (avoids "Connection closed").
# Prerequisites (from linkedin-mcp-server dir):
#   uv run linkedin-mcp-server --get-session   # create LinkedIn session once
#   uv run playwright install chromium         # if you see "Failed to start browser"

uv_path = shutil.which("uv") or "/opt/homebrew/bin/uv"
linkedin_server_dir = os.path.expanduser("~/dev/linkedin-mcp-server")
env = os.environ.copy()
if "/opt/homebrew/bin" not in env.get("PATH", ""):
    env["PATH"] = f"/opt/homebrew/bin:{env.get('PATH', '')}"
env["TRANSPORT"] = "stdio"

li_client = MultiServerMCPClient(
    {
        "linkedin": {
            "command": uv_path,
            "transport": "stdio",
            "args": ["--directory", linkedin_server_dir, "run", "linkedin-mcp-server"],
            "cwd": linkedin_server_dir,
            "env": env,
        }
    }
)
li_tools = await li_client.get_tools()

In [ ]:
# internet search
from tavily import TavilyClient

tavily_client = TavilyClient()

from langchain.tools import tool


@tool
def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search based on the query"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

#  Subagent configurations


In [ ]:
#claude-haiku-4-5-20251001
li_sub_agent = create_agent(
    model = init_chat_model('openai:gpt-4o-mini'),
    #backend = file_system,
    tools = [read_external_file] + li_tools,
    system_prompt=LINKED_IN_PROMPT
)

li_results = await li_sub_agent.ainvoke(
    {
        "messages": [HumanMessage(content="Use Linkedin tools to find  Goodman's Consulting, UK and the director of the company Steven Goodman from Ruislip or London, UK. they should have Goodman's consulting in their professional experience ")]
    }
)

li_results

In [ ]:
li_results["messages"][-1].pretty_print()

In [ ]:
ch_sub_agent = create_agent(
    model = init_chat_model('claude-sonnet-4-5-20250929'),
    #backend = file_system,
    tools = [read_external_file] + ch_tools,
    system_prompt=COMPANIES_HOUSE_PROMPT
)


In [ ]:
results = await li_sub_agent.ainvoke(
    {
        "messages": [HumanMessage(content=application_str)]
    }
)
print(results['messages'][-1].content)

In [ ]:
internet_search_sub_agent = create_agent(
    model = init_chat_model('claude-sonnet-4-5-20250929'),
    #backend = file_system,
    tools = [read_external_file, internet_search],
    system_prompt=INTERNET_SEARCH_PROMPT
)

In [ ]:

# results = await internet_search.ainvoke(
#     {
#         "messages": [HumanMessage(content = application_str)],
        
#     },
#     {
#         "recursion_limit": 5
#     }
# )
print(results["messages"][-1].content)

#### Now configure the subagents for the final graph

In [ ]:
internet_subagent = {
    "name": "internet-search-agent",
    "description": "Use this agent to look for a company website or news about the company.",
    "system_prompt": INTERNET_SEARCH_PROMPT,
    "tools": [read_external_file, internet_search],
    "model": "claude-haiku-4-5-20251001",  
}

linkedin_subagent = {
    "name": "linked-in-search-agent",
    "description": "Use this agent to look to search linked-in for a company page or its employee profiles.",
    "system_prompt": LINKED_IN_PROMPT,
    "tools": [read_external_file] + li_tools,
    "model": "claude-haiku-4-5-20251001",  
}

companies_house_subagent = {
    "name": "companies-house-search-agent",
    "description": "Use this agent to look to search register of incorporated companies.",
    "system_prompt": COMPANIES_HOUSE_PROMPT,
    "tools": [read_external_file] + ch_tools,
    "model": "openai:gpt-4o-mini",  
}
eligibility_subagent = {
    "name": "financial-assessment-search-agent",
    "description": "Use this agent to look to review customer submitted financial documents such as bank statements and annual reports.",
    "model": "claude-sonnet-4-5-20250929",
    "tools":  [read_external_file],
    "system_prompt": ELIGIBILITY_PROMPT
}

financial_assessment_subagent = {
    "name": "financial-assessment-search-agent",
    "description": "Use this agent to look to review customer submitted financial documents such as bank statements and annual reports.",
    "model": "claude-sonnet-4-5-20250929",
    "tools":  [read_external_file],
    "system_prompt": FINANCIAL_ASSESSMENT_PROMPT
}

## Build the orchestrator graph


 1. ingests the application.json or md
 
 2. iterates over the attachments and sends them to the OCR process fo. And the "other OCR process" for RAG doc creation 
 
 3. process application, check policy docs for what is required. 

 4. Deep research  on mcp servers, internet, googlemaps. Results need citations / links to websites, and summarisation of company_report (to be built) 
 and download of the maps to local.




In [ ]:
from typing import List

from langgraph.graph.message import MessagesState


class State(MessagesState):
    application: str
    case_number: str
    documents: List



In [ ]:
from langchain_core.vectorstores import VectorStoreRetriever
from langchain_docling.loader import DoclingLoader
from langchain_openai import OpenAIEmbeddings

from ocr_extraction import DocumentAI


def startup_node(state: State):
    """Process application details and store attributes
    kick off OCR processes and RAG over AnnualReport

    TODOS: some caching so were not rextracting the doc and creating embeddings on every run,
    """
    case_number =state.get("case_number")  # matches the subdirectory under data/
    document_dir = Path("/Users/stevegoodman/dev/fionaa-be/data/") / case_number

    documents =[]
    for document_url in document_dir.iterdir():
        # import json
        # with open("/Users/stevegoodman/dev/fionaa-be/data/workspace/ocr_output/stevejgoodman@gmail.com/5573DraftAccounts_extraction.json", "r") as file:
        #     document = json.load(file)
        document = DocumentAI(document_url, case_number=case_number)
        document.parse()
        document.classify()
        document.extract()
        document.persist()  

        documents.append(document)
    return {"documents" : documents }


In [ ]:
# Set up file system for offloading and access to the policy documents 
# but also want to store LT memories for specifics like the Final Report, and perhaps other things like the Application Form details
# So using composite backend that combines both

from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore

checkpointer=MemorySaver()
store = InMemoryStore()

def make_backend(runtime):
    return CompositeBackend(
        default=StateBackend(runtime),  # Ephemeral storage
        routes={
            "/memories/": StoreBackend(runtime),
            "/disk-files/":  FilesystemBackend(root_dir=str(WORKSPACE),virtual_mode=True)
        }
    )


In [ ]:

################################################## 
# THIS ONE
##################################################


application_assessment = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[read_external_file] + li_tools+ ch_tools,
    store=store,    
    backend=make_backend,
    checkpointer=checkpointer,
    subagents=[eligibility_subagent, financial_assessment_subagent, linkedin_subagent, companies_house_subagent],
    system_prompt = RESEARCH_PROMPT,
)

In [ ]:
builder= StateGraph(State)
builder.add_node("startup", startup_node)

builder.add_node("assessment_deepagent", application_assessment)

builder.add_edge(START, "startup")
builder.add_edge("startup", "assessment_deepagent")


builder.add_edge("assessment_deepagent", END)
fionaa_app = builder.compile(store=store, checkpointer=checkpointer)

In [ ]:
results = await fionaa_app.ainvoke({"messages": [HumanMessage(content=application_str)],
    "case_number": "stevejgoodman@gmail.com"},
    config={"configurable": {"thread_id": "steve3", "recursion_limit": 5}}
)


In [ ]:
results['messages'][-1].pretty_print()

In [ ]:
builder= StateGraph(State)
builder.add_node("startup", startup_node)
builder.add_node("eligibility", eligibility_agent)
builder.add_node("financial_assessment_agent", agent_with_last_message_only(financial_assessment_agent))

builder.add_node("research_agent", research_agent)

builder.add_edge(START, "startup")
builder.add_edge("startup", "eligibility")
builder.add_edge("eligibility", "financial_assessment_agent")
builder.add_edge("financial_assessment_agent", "research_agent")

builder.add_edge("research_agent", END)
fionaa_app = builder.compile(store=store, checkpointer=checkpointer)

In [ ]:
results = await fionaa_app.ainvoke(
    {
        "messages": [HumanMessage(content=application_str)],
        "case_number": "stevejgoodman@gmail.com",  # or another case id
    },
    config={"configurable": {"thread_id": "steve", "recursion_limit": 5}},
)

### Step 4: Test with A User

In [ ]:
print(application_str)

In [ ]:
# Use ainvoke (not invoke): subagent/task tools are async-only; sync .invoke() raises NotImplementedError.
results = await application_assessment.ainvoke(
    {
        "messages": [HumanMessage(content=application_str)],
        "case_number": "stevejgoodman@gmail.com",
    },
    config={"configurable": {"thread_id": "steve", "recursion_limit": 5}},
)
print(results["messages"][-1].content)


In [ ]:

config={"configurable": {"thread_id": "steve", "recursion_limit": 5 }}
results = await application_assessment.ainvoke(
    {
        "messages": [HumanMessage(content=application_str)], 
    },
    config = config
)

In [ ]:
#check that it saved to (In)Memory Store as requested.

In [ ]:
store.get(('filesystem',), '/report.txt').value

# Chatbot Graph

TODO: use agentic RAG
Needs access to the same filesystem in order to get the docs 

In [ ]:


RAG_TEMPLATE = """You are a helpful financial asssitant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
chat_model = ChatOpenAI(model="gpt-4.1-nano")
rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)


def create_vector_store(document_url) -> VectorStoreRetriever:
    loader = DoclingLoader(file_path=document_url)
    raw_docs = loader.load()
    vectorstore = QdrantVectorStore.from_documents(
        raw_docs,
        embeddings,
        location=":memory:",
        collection_name="company_report"
    )
    retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

    return retriever

# run the rag on the business plan/annual report - have no business plans data so use annual report
# state['documents'] = documents
# for document in documents:
#     #print(document.source_document_url)
#     if document["document_type"]=="annual_company_report":
#         retriever = create_vector_store(document["source_document_url"])
#         #state['retriever'] = retriever`




naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | ds, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

naive_retrieval_chain.invoke({"question" : "Provide a short overveiew of this annnual report including this years financial performance v last year"})["response"].content